# 에이전트 워크플로우(Agentic Workflow)

이 노트북은 baseline RAG를 넘어, 질문을 상태(state)로 관리하면서 여러 노드를 순차적으로 실행하는 agentic workflow를 다룬다. 핵심은 모델이 한 번에 답을 찍어내는 것이 아니라, 질문을 정규화하고, 유형을 분류하고, 계획을 세우고, 검색과 도구 호출을 거쳐, 마지막에 근거 검증(grounding verification)까지 수행한다는 점이다.

## 학습 목표
- stateful workflow가 단순 chain과 어떻게 다른지 설명할 수 있다.
- 9개 노드(`normalize_query`부터 `fallback_or_finalize`까지)의 역할, 입력, 출력, 필요성을 이해한다.
- trace 테이블의 `timestamp`, `latency`, `inputs`, `outputs`가 각각 무엇을 의미하는지 읽을 수 있다.
- baseline과 agentic workflow의 차이를 "질문을 해석하고 통제하는 구조" 관점에서 설명할 수 있다.

### 9개 노드 흐름 한눈에 보기
1. `normalize_query`
2. `classify_query`
3. `make_plan`
4. `retrieve_docs`
5. `decide_tools`
6. `run_tools`
7. `synthesize_answer`
8. `verify_grounding`
9. `fallback_or_finalize`

이 순서를 외우기보다, **질문을 정리하고 -> 처리 전략을 정하고 -> 근거를 모으고 -> 답을 만들고 -> 마지막에 검증하고 확정한다**는 큰 흐름으로 이해하는 것이 중요하다.

💡 면접 포인트: stateful workflow를 택한 이유는 각 단계의 입력/출력을 분리해 디버깅과 통제를 가능하게 만들기 위해서라고 설명하면 좋다.


## 개념 설명

첫 코드 셀은 01번 노트북과 마찬가지로 실행 환경을 점검한다. 다만 여기서는 특히 중요하다. workflow 노트북은 여러 모듈을 함께 import하고, 상태 객체와 trace 유틸리티까지 연결하므로 경로가 조금만 어긋나도 뒤 셀이 연쇄적으로 실패할 수 있다.

- **목적**: 올바른 커널과 프로젝트 루트를 확인해, 이후 workflow 셀이 같은 환경에서 안정적으로 이어지게 한다.
- **핵심 로직**: `ROOT`를 조정해 `src/` 모듈을 import 가능하게 만들고, `RuntimeConfig.auto_detect()`로 현재 장치 설정을 함께 출력한다.
- **주요 파라미터/변수**:
  - `ROOT`: 노트북이 의존하는 프로젝트 기준 경로이다.
  - `sys.executable`: 현재 커널의 실제 Python 실행 경로이다.
  - `RuntimeConfig.auto_detect()`: 검색/실행 환경이 `cpu`, `mps`, `cuda` 중 무엇으로 잡혔는지 알려준다.

이 셀의 출력은 기능 셀이 아니지만, 디버깅 비용을 크게 줄여준다. 특히 DGX, 로컬 Mac, CI 환경을 오갈 때 이 확인이 중요하다.

### 실제 구현 펼쳐보기: `_record_node_trace()`
모든 노드가 공통으로 trace를 남기는 핵심 헬퍼는 아래와 같다.

```python
def _record_node_trace(
    state: AgentState,
    node_name: str,
    inputs: dict[str, Any],
    outputs: dict[str, Any],
    start_time: float,
) -> None:
    append_trace(
        state,
        node_name,
        {
            "inputs": inputs,
            "outputs": outputs,
            "latency": round(time.perf_counter() - start_time, 6),
        },
    )
    validate_state(state)
```

핵심 해설:
- 각 노드는 자기 로직을 끝낸 뒤 이 헬퍼를 호출해 `inputs`, `outputs`, `latency`를 한 번에 남긴다.
- 바로 이어서 `validate_state(state)`를 호출한다. 즉, trace 기록과 state post-condition 검사가 항상 함께 간다.
- 이 패턴이 있으면 노드마다 logging 코드를 제각각 작성하지 않아도 된다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## Agentic workflow란 무엇인가

agentic workflow는 하나의 거대한 프롬프트로 모든 문제를 해결하려는 방식이 아니라, 문제를 작은 실행 단계로 분해해 상태를 갱신해 가는 구조이다. 단순 pipeline은 보통 입력이 들어오면 정해진 단계를 무조건 지나간다. 반면 agentic workflow는 **중간 상태를 보고 다음 단계를 조절**하고, 필요하면 tool을 호출하거나, 근거가 약하면 답변을 보류(abstain)할 수 있다.

이 프로젝트의 workflow는 다음 9개 노드로 구성된다.
1. `normalize_query`: 입력을 정리한다.
2. `classify_query`: 질문 유형과 tool 필요 여부를 판단한다.
3. `make_plan`: 질문에 맞는 실행 계획을 만든다.
4. `retrieve_docs`: 관련 문서를 검색한다.
5. `decide_tools`: 어떤 도구를 쓸지 결정한다.
6. `run_tools`: 필요한 도구를 실제 실행한다.
7. `synthesize_answer`: 근거를 바탕으로 초안 답변을 만든다.
8. `verify_grounding`: 답변이 근거 문서와 얼마나 맞는지 검증한다.
9. `fallback_or_finalize`: 충분하면 확정하고, 부족하면 abstain한다.

💡 면접 포인트: "stateful workflow는 reasoning을 잘했다는 느낌보다, 중간 결정과 실패 지점을 추적할 수 있다는 점에서 운영 가능성이 높다"고 설명하면 좋다.

### 읽을 때 체크할 점
- 이 셀의 설명은 **앞 단계 입력이 다음 단계 출력으로 어떻게 이어지는지**를 기준으로 읽으면 훨씬 잘 들어온다.
- 표나 차트가 나오면 숫자만 보지 말고, `왜 이런 값이 나왔는지`를 바로 앞 코드 셀의 함수 호출과 연결해서 해석해보자.


## 구현 준비

이 셀은 실습에 필요한 핵심 모듈을 한 번에 불러온다. 여기서 중요한 것은 각 함수가 서로 다른 책임을 가진다는 점이다. `workflow.py`의 노드 함수들은 상태를 조금씩 갱신하고, `trace_debug`는 그 과정을 사람이 읽을 수 있게 시각화한다.

- **목적**: 이후 셀에서 노드별 동작을 개별적으로 실험할 수 있게 준비한다.
- **핵심 로직**: `build_demo_index(persist=False)`로 retriever를 만들고, 상태 생성 함수와 각 workflow node 함수를 import한다.
- **주요 파라미터/변수**:
  - `retriever`: 문서 검색을 담당하는 객체이다. 이후 `retrieve_docs_node`에 주입된다.
  - `AgentState`, `create_initial_state`: workflow 전체가 공유하는 상태 스키마와 초기 상태 생성 함수이다.
  - `display_trace`: trace를 표 형태로 읽기 쉽게 보여주는 디버깅 도우미이다.

예를 들어:
- `build_demo_index(persist=False)`: 학습용 인덱스를 메모리에 준비한다.
- `from src.workflow import ...`: 노드 함수를 직접 가져오는 이유는 각 단계의 역할을 개별 셀에서 분리해 보여주기 위해서다.

이 노트북은 "하나의 큰 black box 함수"가 아니라 "상태를 바꾸는 작은 함수들의 연쇄"를 이해하는 데 초점을 둔다.

### 설계 포인트
- **목적**: baseline과 달리 여러 함수가 state를 공유하므로, 준비 단계에서 어떤 역할이 어디에 있는지 먼저 잡아야 한다.
- **핵심 로직**: `workflow.py`는 orchestration, `state.py`는 상태 스키마, `trace_debug.py`는 관측성(observability) 담당이다.
- **주요 파라미터**: retriever, 초기 state, trace renderer가 이 노트북의 최소 구성 요소다.


In [ ]:
import pandas as pd

from src.ingestion import build_demo_index
from src.state import AgentState, create_initial_state
from src.trace_debug import display_trace
from src.workflow import (
    classify_query_node,
    decide_tools_node,
    fallback_or_finalize_node,
    make_plan_node,
    normalize_query_node,
    retrieve_docs_node,
    run_tools_node,
    run_workflow,
    synthesize_answer_node,
    verify_grounding_node,
)

retriever = build_demo_index(persist=False)

## 상태 기반 agent 설계(Stateful agent design)

agent가 단순 chain과 다른 가장 큰 이유는 상태(state)를 명시적으로 들고 다닌다는 점이다. 질문을 한 번 받고 바로 답하는 것이 아니라, 중간 결과를 `AgentState` 안에 축적한다. 덕분에 어떤 노드가 어떤 입력을 받고 무엇을 남겼는지 나중에 재구성할 수 있다.

- **목적**: workflow가 어떤 필드를 상태에 저장하는지 전체 그림을 먼저 본다.
- **핵심 로직**: `AgentState.__annotations__`를 표로 바꿔, 상태 필드 이름과 타입을 한눈에 정리한다.
- **주요 파라미터/변수**:
  - `field`: 상태 키 이름이다. 예를 들어 `query_type`, `plan`, `retrieved_docs`, `trace` 등이 있다.
  - `type`: 각 필드가 어떤 타입의 값을 기대하는지 보여준다.
  - `state_schema`: 학습용 상태 스키마 표이다.

이 표를 볼 때는 `final_answer` 같은 최종 산출물뿐 아니라, `draft_answer`, `verification_result`, `errors`, `trace`처럼 운영과 디버깅에 필요한 필드가 왜 필요한지도 같이 생각해보자. 실무에서는 정답을 맞히는 것만큼, 왜 그 답이 나왔는지 설명할 수 있는 구조가 중요하다.

### 실제 구현 펼쳐보기: `AgentState`
agent가 들고 다니는 상태 스키마는 아래와 같다.

```python
class AgentState(TypedDict, total=False):
    session_id: str
    user_query: str
    normalized_query: str
    query_type: str
    requires_tools: bool
    plan: list[str]
    retrieved_docs: list[dict[str, Any]]
    retrieved_memories: list[dict[str, Any]]
    tool_requests: list[dict[str, Any]]
    tool_outputs: list[dict[str, Any]]
    draft_answer: str
    citations: list[dict[str, Any]]
    verification_result: VerificationResult
    final_answer: str
    final_status: str
    trace: list[dict[str, Any]]
    memory_updates: list[dict[str, Any]]
    errors: list[str]
```

필드 읽는 법:
- `session_id`: 실행 단위를 구분하는 식별자
- `user_query`, `normalized_query`: 원본 질문과 정규화된 질문
- `query_type`, `requires_tools`: 분류 단계의 결정
- `plan`: planner가 만든 순서 목록
- `retrieved_docs`, `retrieved_memories`: 외부 문서와 내부 기억 검색 결과
- `tool_requests`, `tool_outputs`: 도구 호출 계획과 실행 결과
- `draft_answer`, `citations`: 합성기 출력
- `verification_result`: verifier의 판단
- `final_answer`, `final_status`: 최종 확정 결과
- `trace`, `memory_updates`, `errors`: 운영/디버깅용 메타데이터

왜 중요한가:
- TypedDict로 필드를 공개해두면, stateful workflow가 `숨겨진 프롬프트 상태`가 아니라 **검사 가능한 데이터 구조**가 된다.


In [ ]:
state_schema = pd.DataFrame(
    {
        'field': list(AgentState.__annotations__.keys()),
        'type': [str(value) for value in AgentState.__annotations__.values()],
    }
)
state_schema

## 노드(node) 개념

이제 첫 번째 실제 노드인 `normalize_query_node`를 실행해본다. node는 상태를 입력으로 받고, 자신이 책임지는 필드만 갱신한 뒤, 그 변화와 메타데이터를 trace에 남긴다. 이런 구조 덕분에 어느 단계에서 무슨 일이 일어났는지 순서대로 복원할 수 있다.

- **목적**: 노드 함수가 상태를 직접 수정한다는 패턴을 체험한다.
- **핵심 로직**: `create_initial_state()`로 빈 상태를 만든 뒤, `normalize_query_node(state)`가 원본 질의를 정리해 `normalized_query` 필드에 저장한다.
- **주요 파라미터/변수**:
  - `state`: workflow 전체에서 공유하는 mutable 상태 객체이다.
  - `user_query`: 사용자가 처음 입력한 질문이다.
  - `normalized_query`: 불필요한 공백/표현 차이를 줄여 후속 단계가 다루기 쉽게 만든 질의이다.

예를 들어 아래 코드에서:
- `create_initial_state('How many days are in the pilot window?')`: 세션 ID, trace, 기본값이 들어간 새 상태를 만든다.
- `normalize_query_node(state)`: 이후 classification과 retrieval이 안정적으로 작동하도록 질문 문자열을 정리한다.

이 노드 자체는 단순해 보이지만, 입력 정규화가 빠지면 같은 의미의 질문도 분기 로직과 검색 점수가 불안정해질 수 있다.

### 실제 구현 펼쳐보기: `create_initial_state()`
초기 state를 만드는 함수는 매우 짧지만 중요하다.

```python
def create_initial_state(user_query: str) -> AgentState:
    return AgentState(
        session_id=str(uuid.uuid4()),
        user_query=user_query,
        trace=[],
        errors=[],
    )
```

주목할 점:
- `session_id`를 UUID로 만든다. 같은 질문을 여러 번 실행해도 세션을 구분할 수 있다.
- `trace`, `errors`를 빈 리스트로 시작한다. 즉, 관측성과 오류 수집이 기본값이다.

💡 면접 포인트: 초기 state에서 trace/errors를 미리 초기화하는 이유는, 디버깅용 필드가 선택 옵션이 아니라 워크플로우 계약(contract)의 일부이기 때문이다.


In [ ]:
state = create_initial_state('How many days are in the pilot window?')
normalize_query_node(state)
pd.Series({'user_query': state['user_query'], 'normalized_query': state['normalized_query']})

## 질의 분류(query classification)

agent workflow가 baseline보다 나아지는 첫 번째 지점은 "질문을 같은 방식으로 처리하지 않는다"는 데 있다. `classify_query_node`는 질문을 `simple_lookup`, `comparison`, `multi_hop`, `summary`, `insufficient_evidence_risk` 중 하나로 분류하고, 도구가 필요한지도 함께 판단한다.

- **목적**: 질문 유형이 후속 plan, retrieval, tool use를 어떻게 바꾸는지 확인한다.
- **핵심 로직**: `classify_query_node(state)`가 정규화된 질문을 읽어 `query_type`과 `requires_tools`를 상태에 기록한다.
- **주요 파라미터/변수**:
  - `query_type`: 실행 전략을 결정하는 핵심 분류 결과이다.
  - `requires_tools`: 계산기나 날짜 파서 같은 로컬 도구가 필요한지 나타내는 불리언 값이다.

이 셀의 출력에서 특히 봐야 할 것은, 날짜 차이를 묻는 질문이 왜 `multi_hop`으로 분류되고 `requires_tools=True`가 되는가이다. retrieval만으로 끝내지 않고 계산 단계를 붙여야 하기 때문이다. 분류가 틀리면 이후 단계가 모두 엇나가므로, classifier는 작아 보여도 전체 품질을 좌우한다.

### 실제 구현 펼쳐보기: `classify_query()`
질문 유형 분류 로직은 아래처럼 완전히 규칙 기반이다.

```python
def classify_query(query: str) -> dict[str, object]:
    normalized = normalize_text(query).lower()

    if any(marker in normalized for marker in INSUFFICIENT_MARKERS):
        return {"query_type": "insufficient_evidence_risk", "requires_tools": False}

    if any(marker in normalized for marker in SUMMARY_MARKERS):
        return {"query_type": "summary", "requires_tools": False}

    if any(marker in normalized for marker in COMPARISON_MARKERS):
        requires_tools = any(marker in normalized for marker in TOOL_MARKERS)
        return {"query_type": "comparison", "requires_tools": requires_tools}

    if any(marker in normalized for marker in MULTI_HOP_MARKERS):
        return {"query_type": "multi_hop", "requires_tools": True}

    return {"query_type": "simple_lookup", "requires_tools": False}
```

라인별 해설:
- `normalize_text(...).lower()`로 비교 전 정규화를 먼저 한다.
- `INSUFFICIENT_MARKERS`를 가장 먼저 체크한다. 순서가 중요하다. 그렇지 않으면 `today's summary` 같은 질문이 summary로 잘못 갈 수 있다.
- `summary`, `comparison`, `multi_hop` 순으로 내려가며 매칭한다.
- 일부 query type은 동시에 `requires_tools=True`도 설정한다.

왜 중요한가:
- classifier는 이후 planner, tool selection, fallback 기준을 모두 좌우하는 첫 관문이다.
- 규칙 기반이라 빠르고 설명 가능하지만, 표현이 복합적이면 오분류 한계가 있다.


In [ ]:
classify_query_node(state)
pd.Series({'query_type': state['query_type'], 'requires_tools': state['requires_tools']})

## 계획(planning)

분류가 끝나면 agent는 곧바로 답하지 않고, 먼저 무엇을 해야 하는지 순서를 잡는다. plan은 reasoning을 길게 출력하는 장식이 아니라, 시스템이 어떤 단계를 거칠지 외부에서 확인 가능하게 만드는 제어 장치이다.

- **목적**: query type에 따라 실행 단계가 어떻게 달라지는지 확인한다.
- **핵심 로직**: `make_plan_node(state)`가 `query_type`을 읽어 ordered step list를 `state['plan']`에 저장한다.
- **주요 파라미터/변수**:
  - `state['plan']`: 사람이 읽을 수 있는 실행 단계 목록이다.
  - `planned_step`: 표로 볼 때 각 행이 하나의 계획 단계가 된다.

예를 들어 아래 코드에서:
- `make_plan_node(state)`: "먼저 검색하고, 필요하면 계산하고, 마지막에 검증한다" 같은 흐름을 명시화한다.
- `pd.DataFrame({'planned_step': state['plan']})`: plan을 리스트가 아니라 표로 보여줘 단계 수와 순서를 읽기 쉽게 만든다.

왜 plan이 필요한가? 문제가 복잡해질수록 agent는 중간에 무엇을 놓쳤는지 설명해야 한다. plan이 있으면 "retrieval을 건너뛰었는가", "tool step이 빠졌는가" 같은 질문에 답하기 쉬워진다.

### 실제 구현 펼쳐보기: `make_plan()`
planner는 query type을 실행 단계 리스트로 바꾼다.

```python
def make_plan(query_type: str) -> list[str]:
    return PLAN_TEMPLATES.get(query_type, PLAN_TEMPLATES["simple_lookup"]).copy()
```

주목할 점:
- `PLAN_TEMPLATES`는 query type별 정적 계획 템플릿이다.
- `simple_lookup`, `comparison`, `multi_hop`, `summary`, `insufficient_evidence_risk`가 각기 다른 step 수를 가진다.
- learned planner가 아니라 **inspectable planner**라서, 왜 이런 경로를 탔는지 바로 설명할 수 있다.

왜 중요한가:
- plan은 chain-of-thought 공개가 아니라 시스템 제어 흐름을 바깥에서 확인 가능하게 만드는 장치다.


In [ ]:
make_plan_node(state)
pd.DataFrame({'planned_step': state['plan']})

## 검색(retrieval)

이제 plan에 따라 실제 근거를 찾는다. `retrieve_docs_node`는 질문과 관련된 청크를 검색해 상태에 저장한다. agentic workflow에서도 retrieval은 여전히 핵심이다. 다만 baseline과 다른 점은, retrieval이 단독 단계가 아니라 **분류와 계획의 결과를 반영한 체계적인 단계**라는 것이다.

- **목적**: 후속 합성에 쓰일 근거 문서를 상태에 적재한다.
- **핵심 로직**: `retrieve_docs_node(state, retriever=retriever, top_k=4)`가 retriever의 `search()`를 호출해 상위 4개 청크를 가져온다.
- **주요 파라미터/변수**:
  - `retriever`: 검색 백엔드 객체이다. `workflow.py`는 `.search()` 인터페이스만 기대하므로 구현체를 바꿔 끼우기 쉽다.
  - `top_k=4`: 상위 4개 청크만 상태에 저장한다.
  - `retrieved_docs`: 각 청크의 `chunk_id`, `source`, `score`, `text`를 담는 리스트이다.

아래 결과 표에서는 `source`가 질문과 맞는 문서를 가리키는지, `score`가 상위 몇 개에 집중되는지, `text` 안에 실제 계산에 필요한 날짜나 수치가 포함되는지를 확인하면 된다. 검색이 틀리면 이후 node가 아무리 정교해도 답은 흔들린다.

### 실제 구현 펼쳐보기: `HybridRetriever.search()` 재참조
검색 자체는 01에서 본 retriever를 그대로 쓴다.

```python
    def search(self, query: str, top_k: int = 5) -> list[dict[str, Any]]:
        if not self.chunks:
            return []

        query_vector = self.vectorizer.transform([query])
        cosine_scores = cosine_similarity(query_vector, self.matrix).flatten()
        lexical_scores = np.array(
            [overlap_ratio(content_tokens(query), content_tokens(chunk["text"])) for chunk in self.chunks]
        )
        combined_scores = (cosine_scores * 0.8) + (lexical_scores * 0.2)
        ranking = np.argsort(combined_scores)[::-1][:top_k]

        results: list[dict[str, Any]] = []
        for index in ranking:
            chunk = dict(self.chunks[index])
            chunk["score"] = round(float(combined_scores[index]), 4)
            results.append(chunk)
        return results
```

여기서 중요한 차이는 검색기 자체보다, **검색이 plan 이후의 의도된 단계**로 들어온다는 점이다. baseline에서는 그냥 질문을 넣고 top-k를 뽑지만, agentic workflow에서는 분류와 계획 결과가 retrieval의 맥락을 만든다.


In [ ]:
retrieve_docs_node(state, retriever=retriever, top_k=4)
pd.DataFrame(state['retrieved_docs'])[['chunk_id', 'source', 'score', 'text']]

## 도구(tools)

질문에 따라서는 문서 검색만으로 충분하지 않다. 날짜 차이나 산술 계산처럼 **문서에서 단서를 찾은 뒤 추가 처리가 필요한 작업**은 tool 단계가 필요하다. 이 workflow에서는 먼저 `decide_tools_node`가 어떤 도구를 쓸지 정하고, 이어서 `run_tools_node`가 실제 도구를 실행한다.

- **목적**: 검색된 근거를 계산 가능한 결과로 변환할 때 필요한 로컬 도구를 연결한다.
- **핵심 로직**: `decide_tools_node(state)`가 `tool_requests`를 만들고, `run_tools_node(state)`가 그 요청을 읽어 `tool_outputs`를 채운다.
- **주요 파라미터/변수**:
  - `tool_requests`: 어떤 도구를 어떤 인자로 호출할지 정의한 구조화된 요청 목록이다.
  - `tool_outputs`: 실제 도구 실행 결과이다.
  - `No tool outputs`: tool이 필요 없는 질문일 때 보여주는 안전한 빈 결과 표시이다.

예를 들어 날짜 차이 질문이라면 classifier가 `requires_tools=True`를 켜고, planner가 계산 단계를 넣은 뒤, tool 단계에서 필요한 계산을 수행한다. 이 설계의 장점은 "도구 선택"과 "도구 실행"을 분리해 디버깅하기 쉽다는 점이다.

### 실제 구현 펼쳐보기: `plan_tool_requests()`
도구 요청은 아래 함수가 결정한다.

```python
def plan_tool_requests(
    query: str,
    query_type: str,
    retrieved_docs: list[dict[str, Any]],
) -> list[ToolRequest]:
    requests: list[ToolRequest] = []
    normalized = normalize_text(query).lower()
    evidence_text = " ".join(doc["text"] for doc in retrieved_docs)
    date_candidates = extract_dates(query) or extract_dates(evidence_text)

    if "how many days" in normalized or "days" in normalized or "며칠" in normalized:
        if len(date_candidates) >= 2:
            requests.append(ToolRequest("date_parser", {"text": date_candidates[0]}))
            requests.append(ToolRequest("date_parser", {"text": date_candidates[1]}))
            requests.append(ToolRequest("calculator", {"expression": "DATE_DIFF_DAYS"}))

    if query_type == "summary":
        requests.append(ToolRequest("keyword_extractor", {"text": evidence_text or query, "limit": 5}))

    return requests
```

왜 필요한가:
- 문서 검색만으로는 날짜 차이, 산술 계산처럼 **문서 단서 + 추가 연산**이 필요한 질문을 처리하기 어렵다.
- 이 단계가 없으면 agent는 계산해야 할 문제도 텍스트 근거만으로 억지로 답하려 할 수 있다.

💡 면접 포인트: tool planning은 LLM function calling 없이도 규칙 기반으로 구현할 수 있고, 이후에 learned router로 확장할 수 있다고 설명하면 좋다.


In [ ]:
decide_tools_node(state)
run_tools_node(state)

pd.DataFrame(state['tool_outputs']) if state['tool_outputs'] else pd.DataFrame([{'message': 'No tool outputs'}])

## 답변 합성(answer synthesis)

이제 agent는 검색 결과와 도구 출력을 종합해 초안 답변(draft answer)을 만든다. 중요한 점은 이 단계가 모든 것을 혼자 결정하지 않는다는 것이다. 이미 앞 단계에서 질문 유형, 계획, 검색 결과, 도구 결과가 준비되어 있으므로, 합성기는 그 재료를 조합하는 역할에 집중한다.

- **목적**: 근거 문서와 도구 결과를 바탕으로 사람이 읽을 수 있는 초안 답변을 생성한다.
- **핵심 로직**: `synthesize_answer_node(state)`가 `retrieved_docs`, `tool_outputs`, `query_type`을 함께 참고해 `draft_answer`와 `citations`를 만든다.
- **주요 파라미터/변수**:
  - `draft_answer`: 아직 검증 전인 초안 답변이다.
  - `citations`: 어떤 문서가 답변에 사용되었는지 추적하기 위한 인용 정보이다.
  - `citation_count`: 인용 문서 개수를 빠르게 확인하는 요약 수치이다.

이 단계가 필요한 이유는 단순히 문서를 나열하는 것만으로는 사용자가 읽을 수 있는 답이 되지 않기 때문이다. 하지만 동시에, 합성 단계는 환각(hallucination)이 끼어들기 쉬운 지점이기도 하다. 그래서 다음 단계인 verifier가 반드시 뒤따라온다.

### 실제 구현 펼쳐보기: `synthesize_answer()`
합성기의 핵심 구현은 아래와 같다.

```python
def synthesize_answer(
    query: str,
    query_type: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]],
) -> dict[str, Any]:
    normalized_query = normalize_text(query).lower()
    ranked = _rank_sentences(query, retrieved_docs)
    evidence_sentences = [sentence for _, sentence, _ in ranked[:3]]
    citations = []
    seen_chunks: set[str] = set()
    for _, _, doc in ranked[:3]:
        if doc["chunk_id"] in seen_chunks:
            continue
        citations.append(
            {
                "doc_id": doc["doc_id"],
                "chunk_id": doc["chunk_id"],
                "source": doc["source"],
                "score": doc["score"],
            }
        )
        seen_chunks.add(doc["chunk_id"])

    if not evidence_sentences:
        draft = "I could not find grounded evidence for this question in the loaded documents."
        return {"draft_answer": draft, "citations": citations}

    calculator_result = next(
        (
            output["output"]["result"]
            for output in tool_outputs
            if output["tool_name"] == "calculator" and output["output"].get("ok")
        ),
        None,
    )
    date_mentions = extract_dates(" ".join(doc["text"] for doc in retrieved_docs))

    if ("how many days" in normalized_query or "days" in normalized_query) and calculator_result is not None:
        if len(date_mentions) >= 2:
            body = f"The pilot window spans {calculator_result} days, from {date_mentions[0]} to {date_mentions[1]}."
        else:
            body = f"The relevant duration is {calculator_result} days."
    elif query_type == "comparison":
        comparison_sentence = next(
            (sentence for sentence in evidence_sentences if "while" in sentence.lower() or "different" in sentence.lower()),
            evidence_sentences[0],
        )
        body = comparison_sentence
    elif query_type == "summary":
        distinct_sentences: list[str] = []
        seen_sources: set[str] = set()
        for _, sentence, doc in ranked:
            if doc["source"] in seen_sources:
                continue
            distinct_sentences.append(sentence)
            seen_sources.add(doc["source"])
            if len(distinct_sentences) == 3:
                break
        body = " ".join(distinct_sentences)
    elif "risk" in normalized_query:
        body = next((sentence for sentence in evidence_sentences if "risk" in sentence.lower()), evidence_sentences[0])
    elif "goal" in normalized_query:
        body = next((sentence for sentence in evidence_sentences if "goal" in sentence.lower()), evidence_sentences[0])
    elif "when" in normalized_query and date_mentions:
        body = next((sentence for sentence in evidence_sentences if extract_dates(sentence)), evidence_sentences[0])
    elif query_type == "multi_hop":
        body = " ".join(evidence_sentences[:2])
    else:
        body = evidence_sentences[0]

    notes = _tool_notes(tool_outputs)
    if notes:
        body = f"{body} {' '.join(notes)}"

    source_list = ", ".join(citation["source"] for citation in citations) or "no sources"
    draft = f"{body} Sources: {source_list}."
    return {"draft_answer": draft, "citations": citations}
```

핵심 해설:
- `_rank_sentences()`로 문장 후보를 다시 정렬한다.
- `query_type`에 따라 summary/comparison/multi-hop 식의 합성 전략이 달라진다.
- `tool_outputs`가 있으면 `_tool_notes()`를 통해 답변 문장에 반영한다.

왜 중요한가:
- agentic workflow는 합성기 혼자 똑똑한 것이 아니라, 앞 단계들이 정리해 준 재료를 조합하는 구조다.


In [ ]:
synthesize_answer_node(state)
pd.Series({'draft_answer': state['draft_answer'], 'citation_count': len(state['citations'])})

## 검증(verification)

`verify_grounding_node`는 초안 답변이 실제로 검색된 문서에 근거하는지 확인한다. 이 단계가 없으면 모델은 문서 밖 지식을 섞거나, 검색 결과에 없는 연결을 스스로 만들어낼 수 있다. 실무에서 RAG 시스템의 신뢰도를 좌우하는 핵심 노드가 바로 근거 검증(grounding verification)이다.

- **목적**: 초안 답변이 검색 근거와 얼마나 맞는지 정량적으로 판단한다.
- **핵심 로직**: `verify_grounding_node(state)`가 답변의 claim을 문서와 대조해 `coverage_score`, `unsupported_claims`, `missing_aspects`, `is_grounded` 등을 계산한다.
- **주요 파라미터/변수**:
  - `coverage_score`: 답변 내용 중 문서로 뒷받침되는 비율에 가까운 값이다.
  - `unsupported_claims`: 문서에서 충분히 찾지 못한 주장 목록이다.
  - `is_grounded`: 전반적으로 근거가 충분하다고 판단했는지 여부이다.

결과를 읽을 때는 `coverage_score`가 높더라도 unsupported claim이 있는지 함께 보아야 한다. 일부 답변은 큰 틀에서는 맞아 보여도 핵심 숫자 하나가 근거 없이 섞여 있을 수 있다.

### 실제 구현 펼쳐보기: `verify_grounding()`
근거 검증 함수는 아래와 같다.

```python
def verify_grounding(
    query: str,
    draft_answer: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]] | None = None,
) -> VerificationResult:
    if not retrieved_docs:
        return VerificationResult(
            is_grounded=False,
            coverage_score=0.0,
            unsupported_claims=["No retrieved evidence"],
            missing_aspects=[query],
        )

    evidence_token_sets = [content_tokens(doc["text"]) for doc in retrieved_docs]
    evidence_union = [token for token_set in evidence_token_sets for token in token_set]
    answer_sentences = [
        sentence
        for sentence in sentence_split(draft_answer)
        if not sentence.lower().startswith("sources:")
    ]

    calculator_tokens = {
        token
        for output in (tool_outputs or [])
        if output["tool_name"] == "calculator" and output["output"].get("ok")
        for token in content_tokens(str(output["output"]["result"]))
    }
    parsed_date_count = sum(
        1
        for output in (tool_outputs or [])
        if output["tool_name"] == "date_parser" and output["output"].get("ok")
    )

    unsupported_claims: list[str] = []
    support_scores: list[float] = []
    for sentence in answer_sentences:
        sentence_tokens = content_tokens(sentence)
        if not sentence_tokens:
            continue
        support = max(overlap_ratio(sentence_tokens, evidence_tokens) for evidence_tokens in evidence_token_sets)
        if calculator_tokens and calculator_tokens.issubset(set(sentence_tokens)) and parsed_date_count >= 2:
            support = max(support, 0.85)
        support_scores.append(support)
        if support < 0.45:
            unsupported_claims.append(sentence)

    query_coverage = overlap_ratio(content_tokens(query), evidence_union)
    answer_coverage = sum(support_scores) / len(support_scores) if support_scores else 0.0
    coverage_score = round(min(1.0, (answer_coverage * 0.75) + (query_coverage * 0.25)), 3)

    missing_aspects: list[str] = []
    if query_coverage < 0.35:
        missing_aspects.append("Evidence only covers part of the request.")

    is_grounded = not unsupported_claims and coverage_score >= 0.6
    return VerificationResult(
        is_grounded=is_grounded,
        coverage_score=coverage_score,
        unsupported_claims=unsupported_claims,
        missing_aspects=missing_aspects,
    )
```

읽는 포인트:
- answer 문장을 sentence 단위로 나눠 각각 support score를 계산한다.
- `coverage_score`는 query coverage와 answer coverage를 섞어 만든다.
- support가 낮은 문장은 `unsupported_claims`로 따로 모은다.

왜 필요한가:
- 이 단계가 없으면 모델이 문서 바깥 지식을 자연스럽게 섞어도 시스템은 모른 척하게 된다.
- verifier는 RAG의 신뢰성을 좌우하는 핵심 방어선이다.

💡 면접 포인트: verifier와 fallback을 분리해 두면, 검증 기준과 최종 정책 결정을 독립적으로 조정할 수 있다.


In [ ]:
verify_grounding_node(state)
pd.Series(state['verification_result'].to_dict())

## fallback 전략

마지막 노드는 검증 결과를 보고 답변을 확정할지, 아니면 abstain할지를 결정한다. agentic workflow가 단순 chain과 다른 중요한 지점이 여기 있다. "모르면 모른다고 말하는 능력"은 지식 시스템에서 성능 저하가 아니라 안전 장치다.

- **목적**: 근거가 충분할 때만 최종 답변을 내보내고, 부족하면 보수적으로 멈춘다.
- **핵심 로직**: `fallback_or_finalize_node(state)`가 `verification_result`와 `query_type`을 읽어 `final_status`와 `final_answer`를 정한다.
- **주요 파라미터/변수**:
  - `final_status`: `answered`, `abstained`, `failed` 같은 최종 상태이다.
  - `final_answer`: 실제 사용자에게 보여줄 최종 문장이다.

왜 필요한가? retrieval이 약하거나 문서 범위 밖 질문이 들어오면, 답변을 억지로 만드는 것보다 abstain하는 편이 훨씬 신뢰할 만하다. 특히 `insufficient_evidence_risk` 유형에서 이 노드의 역할이 두드러진다.

### 실제 구현 펼쳐보기: `fallback_or_finalize()`
최종 답변 확정/보류 로직은 아래 함수에 있다.

```python
def fallback_or_finalize(
    query: str,
    query_type: str,
    draft_answer: str,
    verification_result: VerificationResult,
) -> dict[str, str]:
    if query_type == "insufficient_evidence_risk" and not (
        verification_result.is_grounded
        and verification_result.coverage_score >= 0.85
        and not verification_result.missing_aspects
    ):
        return {
            "final_answer": (
                "The loaded documents do not ground this request, so the workflow is abstaining instead of guessing."
            ),
            "final_status": "abstained",
        }

    if verification_result.is_grounded and verification_result.coverage_score >= 0.65:
        return {"final_answer": draft_answer, "final_status": "answered"}

    explanation = (
        "I do not have enough grounded evidence in the loaded documents to answer this confidently. "
        "Please narrow the question or add more source material."
    )
    if verification_result.missing_aspects:
        explanation = f"{explanation} Missing coverage: {'; '.join(verification_result.missing_aspects)}"

    return {"final_answer": explanation, "final_status": "abstained"}
```

주목할 점:
- `insufficient_evidence_risk`는 더 엄격한 기준으로 본다.
- grounding이 충분하면 `answered`, 아니면 보수적으로 `abstained`를 돌려준다.
- `missing_aspects`를 문장에 넣어, 왜 보류했는지 설명 가능하게 만든다.

왜 필요한가:
- agent는 모르는 것을 그럴듯하게 말하지 않는 쪽이 더 안전하다.
- 실무에서는 false confidence가 단순 무응답보다 더 큰 문제다.


In [ ]:
fallback_or_finalize_node(state)
pd.Series({'final_status': state['final_status'], 'final_answer': state['final_answer']})

## workflow 실행

이제 각 노드를 하나씩 보는 대신, `run_workflow()`로 전체 파이프라인을 끝까지 실행한다. 여기서는 다섯 가지 대표 질문 유형을 모두 돌려보며, classifier가 예상한 타입과 최종 상태가 어떻게 나오는지 비교한다.

- **목적**: 9개 노드가 실제 질문 집합에서 어떻게 조합되어 동작하는지 본다.
- **핵심 로직**: `demo_queries`에 유형별 예시를 넣고, 각 질문에 대해 `run_workflow(question, retriever=retriever)`를 호출해 결과 요약을 누적한다.
- **주요 파라미터/변수**:
  - `expected_demo_type`: 교육용으로 기대하는 질문 유형이다.
  - `predicted_type`: classifier가 실제로 예측한 유형이다.
  - `trace_steps`: 해당 질문이 거친 trace entry 수이다.
  - `final_answer`: 최종적으로 반환된 답변 텍스트이다.

이 표는 "agent가 무엇을 했는가"를 한 줄 요약한 대시보드처럼 읽으면 된다. 특히 `insufficient_evidence_risk` 질문이 실제로 `abstained` 되는지 꼭 확인해보자.

### 읽을 때 체크할 점
- 이 셀의 설명은 **앞 단계 입력이 다음 단계 출력으로 어떻게 이어지는지**를 기준으로 읽으면 훨씬 잘 들어온다.
- 표나 차트가 나오면 숫자만 보지 말고, `왜 이런 값이 나왔는지`를 바로 앞 코드 셀의 함수 호출과 연결해서 해석해보자.


In [ ]:
demo_queries = [
    ('simple_lookup', 'What are the main goals of the workspace policy refresh?'),
    ('comparison', 'How is the rollout plan different from the policy refresh?'),
    ('multi_hop', 'How many days are in the pilot window?'),
    ('summary', 'Summarize the loaded documents.'),
    ('insufficient_evidence_risk', 'Who is the current CEO of the company?'),
]
workflow_runs = []
for expected_type, question in demo_queries:
    result = run_workflow(question, retriever=retriever)
    workflow_runs.append(
        {
            'expected_demo_type': expected_type,
            'predicted_type': result['query_type'],
            'requires_tools': result['requires_tools'],
            'final_status': result['final_status'],
            'trace_steps': len(result['trace']),
            'question': question,
            'final_answer': result['final_answer'],
        }
    )

demo_frame = pd.DataFrame(workflow_runs)
demo_frame

## 실행 추적(trace) 살펴보기

trace는 agentic workflow의 가장 큰 장점 중 하나다. 최종 답변만 보면 왜 그런 결론이 나왔는지 알기 어렵지만, trace를 보면 각 노드가 어떤 입력을 받고 어떤 출력을 만들었는지, 그리고 얼마나 시간이 걸렸는지 단계별로 추적할 수 있다.

- **목적**: happy path 하나를 골라 trace 테이블을 읽는 법을 익힌다.
- **핵심 로직**: `happy_path = run_workflow(...)`로 하나의 상태를 얻고, `display_trace(happy_path['trace'])`로 사람이 읽기 좋은 표로 렌더링한다.
- **주요 파라미터/변수**:
  - `timestamp`: 각 노드가 기록된 시점이다. 실행 순서와 타이밍을 함께 보여준다.
  - `latency`: 해당 노드가 실행되는 데 걸린 시간이다. 상대적으로 느린 구간을 찾는 데 유용하다.
  - `inputs`: 노드가 받은 핵심 입력 요약이다.
  - `outputs`: 노드가 상태에 남긴 주요 결과 요약이다.

trace를 읽을 때는 먼저 node 순서가 기대한 9단계를 따르는지 보고, 그다음 `retrieve_docs`와 `verify_grounding`의 입력/출력이 상식적으로 연결되는지 살펴보자. debug는 항상 "문제가 생긴 마지막 단계"가 아니라 "이상이 시작된 첫 단계"를 찾는 과정이다.

### 실제 구현 펼쳐보기: `append_trace()`와 `validate_state()`
trace 테이블과 post-condition 검사는 아래 구현이 받쳐준다.

```python
def append_trace(state: AgentState, node: str, payload: dict[str, Any]) -> None:
    inputs = payload.get("inputs", {}) if isinstance(payload, dict) else {}
    outputs = payload.get("outputs", payload) if isinstance(payload, dict) else payload
    latency = payload.get("latency") if isinstance(payload, dict) else None
    state.setdefault("trace", []).append(
        {
            "node": node,
            "timestamp": iso_timestamp(),
            "payload": json_ready(payload),
            "inputs": json_ready(inputs),
            "outputs": json_ready(outputs),
            "latency": round(float(latency), 6) if isinstance(latency, (int, float)) else None,
        }
    )
```

```python
def validate_state(state: AgentState) -> None:
    completed_nodes = {str(entry.get("node", "")) for entry in state.get("trace", [])}
    issues: list[str] = []

    for node_name, requirements in NODE_REQUIREMENTS.items():
        if node_name not in completed_nodes:
            continue
        for key, expected_type in requirements.items():
            if key not in state:
                issues.append(f"{key} missing after {node_name}")
                continue
            if _is_invalid(state[key], expected_type):
                issues.append(f"{key} invalid after {node_name}")

    if issues:
        issue_text = "; ".join(issues)
        raise StateValidationError(f"State validation failed: {issue_text}")
```

trace 읽는 법:
- `step`: 실행 순서
- `node`: 어떤 노드가 돌았는지
- `latency`: 해당 단계 지연 시간
- `inputs`: 그 노드가 실제로 받은 입력
- `outputs`: 그 노드가 남긴 결과

왜 중요한가:
- trace logging이 있어야 `왜 이 답이 나왔는가`를 거꾸로 추적할 수 있다.
- state validation이 있어야 어떤 노드가 필수 필드를 안 채웠는지 바로 잡아낸다.

💡 면접 포인트: observability를 처음부터 설계에 넣어야 운영 시 디버깅 비용이 급격히 줄어든다.


In [ ]:
happy_path = run_workflow('How many days are in the pilot window?', retriever=retriever)
display_trace(happy_path['trace'])

## 실험

이 셀은 전체 실행 결과에서 핵심 컬럼만 다시 뽑아, 질문 유형별 동작 차이를 더 선명하게 보여준다. 복잡한 전체 결과를 그대로 읽기 어렵다면, 먼저 필요한 열만 남기는 것이 분석의 기본이다.

- **목적**: classifier 예측, tool 필요 여부, final status, trace step 수를 중심으로 비교한다.
- **핵심 로직**: `demo_frame`에서 핵심 열만 선택해 표시한다.
- **주요 파라미터/변수**:
  - `requires_tools`: 질문마다 tool 단계가 실제로 필요한지를 보여준다.
  - `trace_steps`: 질문 유형에 따라 workflow가 비슷한 단계 수를 가지는지 확인하는 보조 지표이다.
  - `question`: 어떤 질문이 어떤 실행 패턴을 만들었는지 연결하는 기준 열이다.

이 표를 볼 때는 `comparison`과 `multi_hop`처럼 reasoning 부담이 있는 질문이 정말로 더 복잡한 경로를 타는지, 그리고 insufficient evidence 질문이 answered가 아니라 abstained로 가는지 확인해보자.

### 읽을 때 체크할 점
- 이 셀의 설명은 **앞 단계 입력이 다음 단계 출력으로 어떻게 이어지는지**를 기준으로 읽으면 훨씬 잘 들어온다.
- 표나 차트가 나오면 숫자만 보지 말고, `왜 이런 값이 나왔는지`를 바로 앞 코드 셀의 함수 호출과 연결해서 해석해보자.


In [ ]:
demo_frame[['expected_demo_type', 'predicted_type', 'requires_tools', 'final_status', 'trace_steps', 'question']]

## 결과 해석

이 마지막 분석 셀은 예측된 질문 유형과 최종 상태별로 평균 trace step 수를 묶어 보여준다. step 수 자체가 품질을 의미하지는 않지만, agent가 질문에 따라 얼마나 다른 제어 흐름을 밟았는지를 읽는 데 도움이 된다.

- **목적**: workflow가 모든 질문을 획일적으로 처리하지 않는다는 점을 수치로 확인한다.
- **핵심 로직**: `groupby(['predicted_type', 'final_status'])['trace_steps'].mean()`으로 유형-상태 조합별 평균 단계를 계산한다.
- **주요 파라미터/변수**:
  - `predicted_type`: classifier의 실제 라우팅 결과이다.
  - `final_status`: answered / abstained 같은 최종 제어 결과이다.
  - `trace_steps`: 해당 조합에서 평균적으로 몇 개의 노드 기록이 남는지 보여준다.

결과를 읽을 때는 step 수가 조금 다르다고 해서 무조건 좋은 것은 아니다. 중요한 것은 "필요할 때만 복잡해지는가"이다. 예를 들어 abstain 경로는 검증 뒤 조기에 종료될 수 있고, tool이 필요한 질문은 중간 단계가 더 풍부해질 수 있다.

baseline과 비교하면, agentic workflow의 본질은 답변 길이나 문장 스타일이 아니라 **질문을 상태와 제어 흐름으로 다룬다**는 점이다.

baseline(01)과 비교해 여기서 달라진 점은 `검색만 잘하면 된다`가 아니라 `질문 분류, 계획, 도구, 검증, fallback까지 포함한 제어 시스템`이 되었다는 점이다. 즉, agentic workflow는 단순한 길어진 chain이 아니라 **조건 분기와 상태 추적이 가능한 state machine에 가깝다**.


In [ ]:
demo_frame.groupby(['predicted_type', 'final_status'])['trace_steps'].mean().reset_index()

## 핵심 정리

이 노트북을 통해 agentic workflow가 단순 chain보다 왜 설명 가능하고 운영 친화적인지 확인했다. 9개 노드는 각자 작은 책임을 가지며, 상태와 trace를 통해 서로 연결된다. 그 결과 어떤 질문이 어떤 경로를 탔는지, 왜 tool이 호출되었는지, 왜 abstain했는지를 단계별로 설명할 수 있다.

특히 `verify_grounding`과 `fallback_or_finalize`는 실무형 agent 설계에서 매우 중요한 차별점이다. 답변을 만드는 능력만큼, 부족한 근거를 감지하고 멈추는 능력이 신뢰도를 좌우한다.

💡 면접 포인트: "baseline은 retrieve하고 바로 answer를 만들지만, agentic workflow는 classify-plan-retrieve-tool-synthesize-verify-fallback의 제어 루프를 통해 질문을 더 안전하게 다룬다"고 정리하면 좋다.

면접에서는 `왜 stateful workflow를 선택했는가`, `왜 verifier와 fallback이 필요한가`, `왜 trace를 남겼는가` 세 질문에 답할 수 있으면 이 노트북의 핵심을 제대로 설명한 것이다.
